# 🏠 Private Property Scraping Pipeline

This notebook builds the scraping pipeline for Private Property listings and rental data.

The goal is to extract structured property data directly from listing pages and store it in a format that can be used downstream for cleaning, analysis, and financial modelling.

The pipeline focuses on:

Extracting listing-level attributes (price, location, features)
Standardising fields across listings
Handling inconsistencies in raw HTML structures
Producing structured datasets for both sales and rentals


## 1) Install dependencies

Run the pip cell only if these packages are not already installed in your environment.

In [1]:
# Uncomment only if needed
%pip install pandas numpy requests beautifulsoup4 lxml tqdm

Note: you may need to restart the kernel to use updated packages.


## 2) Imports and Setup

In [2]:
import json
import logging
import random
import re
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional
from urllib.parse import parse_qsl, urlencode, urlparse, urlunparse

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

## 3) Create project directories

In [3]:
DIRS = [
    "data/raw/listings",
    "data/raw/rentals",
    "data/raw/area_data",
    "data/interim",
    "data/processed",
    "logs",
]

for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✅ Directories ready")
for d in DIRS:
    print(" -", d)

✅ Directories ready
 - data/raw/listings
 - data/raw/rentals
 - data/raw/area_data
 - data/interim
 - data/processed
 - logs


## 4) Logging and scraper configuration

The `SEARCH_SEEDS` below use **Private Property page names and area ids exactly as they appear on the website**.  
This section defines base URLs, headers, and core parameters used throughout the scraping process.

We mimic a browser request to reduce blocking and improve response consistency.

In [4]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("private_property")

@dataclass
class ScraperConfig:
    max_sale_records: int = 1000
    max_rental_records: int = 1000
    max_pages_per_seed: int = 30
    request_timeout: int = 30
    min_delay_seconds: float = 2.2
    max_delay_seconds: float = 4.0
    detail_pause_every: int = 40
    long_pause_seconds: float = 12.0
    user_agent: str = (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )

CONFIG = ScraperConfig()

SEARCH_SEEDS = {
    "sale": [
        {
            "property_type_seed": "House",
            "seed_name": "Johannesburg Metro Houses For Sale",
            "seed_url": "https://www.privateproperty.co.za/houses-for-sale/johannesburg-metro/33",
        },
        {
            "property_type_seed": "Apartment",
            "seed_name": "Johannesburg Metro Apartments For Sale",
            "seed_url": "https://www.privateproperty.co.za/apartments-for-sale/johannesburg-metro/33",
        },
        {
            "property_type_seed": "House",
            "seed_name": "Pretoria Houses For Sale",
            "seed_url": "https://www.privateproperty.co.za/houses-for-sale/pretoria/28",
        },
        {
            "property_type_seed": "Apartment",
            "seed_name": "Pretoria Apartments For Sale",
            "seed_url": "https://www.privateproperty.co.za/apartments-for-sale/pretoria/28",
        },
        {
            "property_type_seed": "House",
            "seed_name": "Centurion Houses For Sale",
            "seed_url": "https://www.privateproperty.co.za/houses-for-sale/centurion/32",
        },
        {
            "property_type_seed": "Apartment",
            "seed_name": "Centurion Apartments For Sale",
            "seed_url": "https://www.privateproperty.co.za/apartments-for-sale/centurion/32",
        },
    ],
    "rent": [
        {
            "property_type_seed": "House",
            "seed_name": "Johannesburg Metro Houses To Rent",
            "seed_url": "https://www.privateproperty.co.za/houses-to-rent/johannesburg-metro/33",
        },
        {
            "property_type_seed": "Apartment",
            "seed_name": "Johannesburg Metro Apartments To Rent",
            "seed_url": "https://www.privateproperty.co.za/apartments-to-rent/johannesburg-metro/33",
        },
        {
            "property_type_seed": "House",
            "seed_name": "Pretoria Houses To Rent",
            "seed_url": "https://www.privateproperty.co.za/houses-to-rent/pretoria/28",
        },
        {
            "property_type_seed": "Apartment",
            "seed_name": "Pretoria Apartments To Rent",
            "seed_url": "https://www.privateproperty.co.za/apartments-to-rent/pretoria/28",
        },
        {
            "property_type_seed": "House",
            "seed_name": "Centurion Houses To Rent",
            "seed_url": "https://www.privateproperty.co.za/houses-to-rent/centurion/32",
        },
        {
            "property_type_seed": "Apartment",
            "seed_name": "Centurion Apartments To Rent",
            "seed_url": "https://www.privateproperty.co.za/apartments-to-rent/centurion/32",
        },
    ],
}

SALE_DETAIL_RE = re.compile(r"/for-sale/.+/(T\d+)(?:$|[?#])", flags=re.I)
RENT_DETAIL_RE = re.compile(r"/to-rent/.+/(RR\d+)(?:$|[?#])", flags=re.I)

## 5) Core helpers

In [5]:
def jitter_sleep(min_seconds: float, max_seconds: float) -> None:
    time.sleep(random.uniform(min_seconds, max_seconds))

def clean_text(value: Optional[str]) -> Optional[str]:
    if value is None:
        return None
    value = str(value)
    value = (
        value.replace("\xa0", " ")
             .replace("\u202f", " ")
             .replace("\u2009", " ")
             .replace("\u2007", " ")
    )
    value = re.sub(r"\s+", " ", value).strip()
    return value or None

def title_case_slug(value: Optional[str]) -> Optional[str]:
    value = clean_text(value)
    if value is None:
        return None
    return value.replace("-", " ").replace("_", " ").title()

def parse_float(value: Optional[str]) -> Optional[float]:
    value = clean_text(value)
    if value is None:
        return None
    value = value.replace(",", "")
    match = re.search(r"-?\d+(?:\.\d+)?", value)
    if not match:
        return None
    try:
        return float(match.group(0))
    except Exception:
        return None

def parse_int(value: Optional[str]) -> Optional[int]:
    parsed = parse_float(value)
    if parsed is None:
        return None
    return int(round(parsed))

def parse_money(value: Optional[str]) -> Optional[float]:
    value = clean_text(value)
    if value is None:
        return None

    raw = (
        value.replace("\xa0", " ")
             .replace("\u202f", " ")
             .replace("\u2009", " ")
             .replace("\u2007", " ")
    )

    patterns = [
        r"R\s*([0-9][0-9\s,\.]{3,})",
        r"([0-9]{1,3}(?:[\s,][0-9]{3})+(?:\.\d+)?)",
        r"([0-9]+(?:\.\d+)?)",
    ]

    for pattern in patterns:
        match = re.search(pattern, raw, flags=re.I)
        if not match:
            continue
        candidate = match.group(1)
        candidate = re.sub(r"[^0-9.]", "", candidate)
        if not candidate:
            continue
        try:
            amount = float(candidate)
            if amount > 0:
                return amount
        except Exception:
            continue

    numeric = re.sub(r"[^0-9.]", "", raw)
    if not numeric:
        return None
    try:
        amount = float(numeric)
        return amount if amount > 0 else None
    except Exception:
        return None

def first_non_null(*values):
    for value in values:
        if value is None:
            continue
        if isinstance(value, str) and clean_text(value) is None:
            continue
        return value
    return None

def unique_preserve_order(values: Iterable[str]) -> list[str]:
    seen = set()
    out = []
    for value in values:
        if value and value not in seen:
            seen.add(value)
            out.append(value)
    return out

def html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    text = soup.get_text("\n", strip=True)
    text = text.replace("\xa0", " ").replace("\u202f", " ").replace("\u2009", " ")
    text = re.sub(r"\n{2,}", "\n", text)
    return text

def build_results_url(seed_url: str, page: int) -> str:
    parsed = urlparse(seed_url)
    query = dict(parse_qsl(parsed.query, keep_blank_values=True))
    if page > 1:
        query["page"] = str(page)
    else:
        query.pop("page", None)
    return urlunparse(parsed._replace(query=urlencode(query)))

def extract_listing_id_from_url(url: Optional[str]) -> Optional[str]:
    url = clean_text(url)
    if url is None:
        return None
    match = re.search(r"/((?:T|RR)\d+)(?:$|[?#])", url, flags=re.I)
    return match.group(1).upper() if match else None

def normalize_property_type(raw: Optional[str]) -> Optional[str]:
    raw = clean_text(raw)
    if raw is None:
        return None
    low = raw.lower()
    if "apartment" in low or "flat" in low or "studio" in low:
        return "Apartment"
    if "house" in low:
        return "House"
    return None

def split_locality(locality: Optional[str]) -> tuple[Optional[str], Optional[str]]:
    locality = clean_text(locality)
    if locality is None:
        return None, None
    parts = [clean_text(x) for x in locality.split(",") if clean_text(x)]
    if not parts:
        return None, None
    suburb = parts[0]
    city = parts[1] if len(parts) > 1 else None
    return suburb, city

def parse_detail_location_from_url(url: str) -> dict:
    parsed = urlparse(url)
    parts = [p for p in parsed.path.strip("/").split("/") if p]
    out = {
        "pp_transaction_slug": None,
        "pp_province_slug": None,
        "pp_metro_slug": None,
        "pp_city_slug": None,
        "pp_suburb_slug": None,
    }
    if len(parts) >= 2:
        out["pp_transaction_slug"] = parts[0]
        out["pp_province_slug"] = parts[1]
    if len(parts) >= 3:
        out["pp_metro_slug"] = parts[2]
    if len(parts) >= 4:
        out["pp_city_slug"] = parts[3]
    if len(parts) >= 5:
        out["pp_suburb_slug"] = parts[4]
    return out

def regex_group(text: str, pattern: str, flags: int = re.I) -> Optional[str]:
    match = re.search(pattern, text, flags)
    if not match:
        return None
    for idx in range(1, (match.lastindex or 0) + 1):
        value = clean_text(match.group(idx))
        if value is not None:
            return value
    return None


## 6) HTTP session setup

In [6]:
def build_session(user_agent: str) -> requests.Session:
    session = requests.Session()
    retries = Retry(
        total=4,
        connect=4,
        read=4,
        backoff_factor=1.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update(
        {
            "User-Agent": user_agent,
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-ZA,en;q=0.9",
            "Cache-Control": "no-cache",
            "Pragma": "no-cache",
        }
    )
    return session

def fetch_html(session: requests.Session, url: str, cfg: ScraperConfig) -> str:
    logger.info(f"GET {url}")
    response = session.get(url, timeout=cfg.request_timeout)
    response.raise_for_status()
    jitter_sleep(cfg.min_delay_seconds, cfg.max_delay_seconds)
    return response.text

## 7) Result-page extraction

This step collects **detail-page URLs** from each search page.

### Why this is stable
Private Property result pages expose listing detail URLs directly in the HTML and also embed listing metadata in JSON-LD blocks.  
That means I can scrape without Selenium and without relying on fragile browser automation.

In [7]:
def extract_jsonld_objects(soup: BeautifulSoup) -> list[dict]:
    objects = []
    for script in soup.find_all("script", type=lambda x: x and "ld+json" in str(x)):
        raw = script.get_text(" ", strip=True)
        if not raw:
            continue
        try:
            obj = json.loads(raw)
            if isinstance(obj, dict):
                objects.append(obj)
            elif isinstance(obj, list):
                objects.extend([x for x in obj if isinstance(x, dict)])
        except Exception:
            continue
    return objects

def extract_result_links(html: str, mode: str) -> list[str]:
    soup = BeautifulSoup(html, "lxml")
    urls = []

    for obj in extract_jsonld_objects(soup):
        if obj.get("@type") != "Residence":
            continue
        url = clean_text(obj.get("url"))
        if url:
            urls.append(url.split("?")[0].rstrip("/"))

    pattern = SALE_DETAIL_RE if mode == "sale" else RENT_DETAIL_RE
    for anchor in soup.find_all("a", href=True):
        href = clean_text(anchor["href"])
        if href is None:
            continue
        href = href.split("?")[0].rstrip("/")
        if href.startswith("/"):
            href = "https://www.privateproperty.co.za" + href
        if pattern.search(href):
            urls.append(href)

    return unique_preserve_order(urls)

def collect_detail_urls_for_mode(mode: str, cfg: ScraperConfig) -> list[str]:
    assert mode in {"sale", "rent"}
    target = cfg.max_sale_records if mode == "sale" else cfg.max_rental_records

    session = build_session(cfg.user_agent)
    all_urls = []
    seen = set()

    for seed in SEARCH_SEEDS[mode]:
        if len(all_urls) >= target:
            break

        logger.info("=" * 70)
        logger.info(f"{mode.upper()} seed: {seed['seed_name']}")

        empty_pages = 0

        for page in range(1, cfg.max_pages_per_seed + 1):
            if len(all_urls) >= target:
                break

            results_url = build_results_url(seed["seed_url"], page)
            html = fetch_html(session, results_url, cfg)
            links = extract_result_links(html, mode=mode)

            new_links = [link for link in links if link not in seen]

            if not new_links:
                empty_pages += 1
                logger.info(f"Page {page}: 0 new links")
                if empty_pages >= 2:
                    logger.info("Stopping this seed after consecutive empty pages")
                    break
                continue

            empty_pages = 0

            for link in new_links:
                seen.add(link)
                all_urls.append(link)
                if len(all_urls) >= target:
                    break

            logger.info(
                f"Page {page}: collected {len(new_links)} new links "
                f"(running total={len(all_urls)})"
            )

    return all_urls[:target]

## 8) Detail-page parsing

Each detail page contains the most reliable fields for the dataset:
- listing number
- property type
- listing date
- floor size
- land size
- levies
- rates and taxes
- bedrooms
- bathrooms
- garage / covered / open parking
- price or monthly rent

The parser below is layered:
1. URL path
2. page title and visible text
3. JSON-LD
4. regex fallbacks

In [8]:
def extract_residence_jsonld(html: str) -> Optional[dict]:
    soup = BeautifulSoup(html, "lxml")
    for obj in extract_jsonld_objects(soup):
        if obj.get("@type") == "Residence":
            return obj
    return None

def extract_meta_description(soup: BeautifulSoup) -> Optional[str]:
    tag = soup.find("meta", attrs={"name": "description"})
    if tag and tag.get("content"):
        return clean_text(tag.get("content"))
    return None

def extract_price_from_detail_page(html: str, soup: BeautifulSoup, page_text: str, mode: str) -> Optional[float]:
    candidates = []

    # 1) Plain-text anchors visible on the page
    patterns = []
    if mode == "sale":
        patterns = [
            r"^\s*(R\s*[0-9\s\u00a0\u202f,\.]{3,})\s*$",
            r"(R\s*[0-9\s\u00a0\u202f,\.]{3,})\s*calculate bond costs",
            r"(R\s*[0-9\s\u00a0\u202f,\.]{3,})\s*#",
            r"(R\s*[0-9\s\u00a0\u202f,\.]{3,})",
        ]
    else:
        patterns = [
            r"(R\s*[0-9\s\u00a0\u202f,\.]{3,}\s*(?:Per Month|per month|pm))",
            r"(R\s*[0-9\s\u00a0\u202f,\.]{3,})",
        ]

    for pattern in patterns:
        m = re.search(pattern, page_text, flags=re.I | re.M)
        if m:
            candidates.append(m.group(1))

    # 2) Check meta tags commonly used for share cards / SEO
    meta_keys = [
        ("property", "product:price:amount"),
        ("property", "og:price:amount"),
        ("name", "twitter:data1"),
        ("itemprop", "price"),
        ("property", "price"),
    ]
    for attr_name, attr_value in meta_keys:
        tag = soup.find("meta", attrs={attr_name: attr_value})
        if tag and tag.get("content"):
            candidates.append(tag.get("content"))

    # 3) Raw HTML fallback around the first currency occurrence
    html_clean = (
        html.replace("\xa0", " ")
            .replace("\u202f", " ")
            .replace("\u2009", " ")
            .replace("\u2007", " ")
    )
    html_match = re.search(r"R\s*[0-9][0-9\s,\.]{3,}", html_clean, flags=re.I)
    if html_match:
        candidates.append(html_match.group(0))

    # 4) Description fallback as a last resort
    description = extract_meta_description(soup)
    if description:
        candidates.append(description)

    for candidate in candidates:
        amount = parse_money(candidate)
        if amount is None:
            continue
        if mode == "sale" and amount >= 100000:
            return amount
        if mode == "rent" and 500 <= amount <= 1000000:
            return amount

    return None

def parse_detail_page(html: str, url: str, mode: str, seed_property_type: Optional[str] = None) -> dict:
    soup = BeautifulSoup(html, "lxml")
    page_text = html_to_text(html)
    residence = extract_residence_jsonld(html) or {}
    address = residence.get("address", {}) if isinstance(residence, dict) else {}
    locality = address.get("addressLocality") if isinstance(address, dict) else None
    region = address.get("addressRegion") if isinstance(address, dict) else None

    title_h1 = soup.find("h1")
    title = clean_text(title_h1.get_text(" ", strip=True) if title_h1 else None)
    if title is None and soup.title:
        title = clean_text(soup.title.get_text(" ", strip=True))

    description = extract_meta_description(soup)

    url_loc = parse_detail_location_from_url(url)
    locality_suburb, locality_city = split_locality(locality)

    suburb = first_non_null(title_case_slug(url_loc.get("pp_suburb_slug")), locality_suburb)
    city = first_non_null(title_case_slug(url_loc.get("pp_city_slug")), locality_city)
    province = first_non_null(title_case_slug(url_loc.get("pp_province_slug")), clean_text(region))

    property_type = first_non_null(
        normalize_property_type(regex_group(page_text, r"Property type\s+([A-Za-z ]+)")),
        normalize_property_type(title),
        normalize_property_type(seed_property_type),
    )

    title = first_non_null(
        title,
        f"{property_type or 'Property'} in {suburb}" if suburb else None,
    )

    listing_id = first_non_null(
        regex_group(page_text, r"Listing number\s+((?:T|RR)\d+)"),
        extract_listing_id_from_url(url),
    )

    listing_date = regex_group(page_text, r"Listing date\s+([0-9]{1,2}\s+[A-Za-z]{3}\s+[0-9]{4})")
    floor_area_sqm = parse_float(regex_group(page_text, r"Floor size\s+([0-9 ,\.]+)\s*m²"))
    land_area_sqm = parse_float(regex_group(page_text, r"Land size\s+([0-9 ,\.]+)\s*m²"))
    levies = parse_money(regex_group(page_text, r"Levies\s+R?\s*([0-9 ,\.]+)"))
    rates_taxes = parse_money(regex_group(page_text, r"Rates and taxes\s+R?\s*([0-9 ,\.]+)"))

    bedrooms = parse_float(regex_group(page_text, r"Bedrooms\s+([0-9.]+)"))
    bathrooms = parse_float(regex_group(page_text, r"Bathrooms\s+([0-9.]+)"))
    garage = parse_float(
        first_non_null(
            regex_group(page_text, r"Garage parking\s+([0-9.]+)"),
            regex_group(page_text, r"Garages\s+([0-9.]+)"),
        )
    )
    covered_parking = parse_float(regex_group(page_text, r"Covered parking\s+([0-9.]+)"))
    open_parking = parse_float(regex_group(page_text, r"Open parking\s+([0-9.]+)"))
    parking_spaces = sum(x for x in [garage, covered_parking, open_parking] if x is not None) or None

    available_text = first_non_null(
        regex_group(page_text, r"(Available\s+Now)"),
        regex_group(page_text, r"(Available\s+[0-9]{1,2}\s+[A-Za-z]+\s+[0-9]{4})"),
        regex_group(page_text, r"(Available\s+[A-Za-z]+\s+[0-9]{4})"),
    )

    furnished_flag = None
    low_text = page_text.lower()
    if "unfurnished" in low_text:
        furnished_flag = 0
    elif "furnished" in low_text:
        furnished_flag = 1

    amount = extract_price_from_detail_page(html, soup, page_text, mode=mode)

    if mode == "sale":
        price_field_name = "purchase_price"
        url_field_name = "listing_url"
    else:
        price_field_name = "monthly_rent"
        url_field_name = "rental_url"

    row = {
        "source_site": "privateproperty",
        "listing_id": listing_id,
        url_field_name: url,
        "title": title,
        price_field_name: amount,
        "suburb": suburb,
        "city": city,
        "province": province,
        "property_type": property_type,
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "parking_spaces": parking_spaces,
        "garage": garage,
        "floor_area_sqm": floor_area_sqm,
        "land_area_sqm": land_area_sqm,
        "levies": levies,
        "rates_taxes": rates_taxes,
        "description": description,
        "listing_date": listing_date,
        "availability_text": available_text if mode == "rent" else None,
        "furnished_flag": furnished_flag if mode == "rent" else None,
        "scraped_timestamp": pd.Timestamp.utcnow().isoformat(),
        "pp_transaction_slug": url_loc.get("pp_transaction_slug"),
        "pp_province_slug": url_loc.get("pp_province_slug"),
        "pp_metro_slug": url_loc.get("pp_metro_slug"),
        "pp_city_slug": url_loc.get("pp_city_slug"),
        "pp_suburb_slug": url_loc.get("pp_suburb_slug"),
    }
    return row


In [9]:
# Quick parser smoke test on one known sale page before running the full scrape

test_url = "https://www.privateproperty.co.za/for-sale/gauteng/johannesburg-metro/sandton/bryanston/T5382012"

test_session = build_session(CONFIG.user_agent)
test_html = fetch_html(test_session, test_url, CONFIG)
test_row = parse_detail_page(test_html, test_url, mode="sale", seed_property_type="House")

print("Smoke test purchase_price:", test_row.get("purchase_price"))
display(
    pd.DataFrame([test_row])[
        ["listing_id", "purchase_price", "suburb", "city", "property_type", "bedrooms", "bathrooms"]
    ]
)

17:26:51  [INFO]  GET https://www.privateproperty.co.za/for-sale/gauteng/johannesburg-metro/sandton/bryanston/T5382012


Smoke test purchase_price: 19500000.0


,listing_id,purchase_price,suburb,city,property_type,bedrooms,bathrooms
0,T5382012,19500000.0,Bryanston,Sandton,House,5.0,5.5


## 9) Dataset schemas and validation

In [10]:
LISTINGS_COLS = [
    "source_site",
    "listing_id",
    "listing_url",
    "title",
    "purchase_price",
    "suburb",
    "city",
    "province",
    "property_type",
    "bedrooms",
    "bathrooms",
    "parking_spaces",
    "garage",
    "floor_area_sqm",
    "land_area_sqm",
    "levies",
    "rates_taxes",
    "description",
    "listing_date",
    "scraped_timestamp",
    "pp_transaction_slug",
    "pp_province_slug",
    "pp_metro_slug",
    "pp_city_slug",
    "pp_suburb_slug",
]

RENTALS_COLS = [
    "source_site",
    "listing_id",
    "rental_url",
    "title",
    "monthly_rent",
    "suburb",
    "city",
    "province",
    "property_type",
    "bedrooms",
    "bathrooms",
    "parking_spaces",
    "garage",
    "floor_area_sqm",
    "land_area_sqm",
    "levies",
    "rates_taxes",
    "furnished_flag",
    "availability_text",
    "description",
    "listing_date",
    "scraped_timestamp",
    "pp_transaction_slug",
    "pp_province_slug",
    "pp_metro_slug",
    "pp_city_slug",
    "pp_suburb_slug",
]

def coerce_nulls(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    return df.replace({"": np.nan, "None": np.nan, "none": np.nan, "null": np.nan, "Null": np.nan})

def dedupe_frame(df: pd.DataFrame, mode: str) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    url_col = "listing_url" if mode == "sale" else "rental_url"
    subset = [c for c in ["listing_id", url_col] if c in df.columns]
    return df.drop_duplicates(subset=subset, keep="first").reset_index(drop=True)

def standardize_location_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    for col in ["suburb", "city", "province"]:
        if col in df.columns:
            df[col] = df[col].map(lambda x: title_case_slug(x) if pd.notna(x) else x)
    return df

def validate_numeric_ranges(df: pd.DataFrame, mode: str) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    work = df.copy()

    numeric_cols = [
        "bedrooms", "bathrooms", "parking_spaces", "garage",
        "floor_area_sqm", "land_area_sqm", "levies", "rates_taxes",
    ]
    for col in numeric_cols:
        if col in work.columns:
            work[col] = pd.to_numeric(work[col], errors="coerce")

    if mode == "sale":
        work["purchase_price"] = pd.to_numeric(work["purchase_price"], errors="coerce")
        work = work[(work["purchase_price"].isna()) | (work["purchase_price"] > 0)]
    else:
        work["monthly_rent"] = pd.to_numeric(work["monthly_rent"], errors="coerce")
        work = work[(work["monthly_rent"].isna()) | (work["monthly_rent"] > 0)]

    bounds = {
        "bedrooms": (0, 20),
        "bathrooms": (0, 20),
        "parking_spaces": (0, 20),
        "garage": (0, 20),
        "floor_area_sqm": (0, 100000),
        "land_area_sqm": (0, 500000),
        "levies": (0, 100000),
        "rates_taxes": (0, 100000),
    }

    for col, (lo, hi) in bounds.items():
        if col in work.columns:
            work = work[(work[col].isna()) | ((work[col] >= lo) & (work[col] <= hi))]

    if "property_type" in work.columns:
        work = work[(work["property_type"].isna()) | (work["property_type"].isin(["House", "Apartment"]))]

    return work.reset_index(drop=True)

def finalize_frame(df: pd.DataFrame, mode: str) -> pd.DataFrame:
    columns = LISTINGS_COLS if mode == "sale" else RENTALS_COLS

    work = df.copy()
    work = coerce_nulls(work)
    work = dedupe_frame(work, mode=mode)
    work = standardize_location_columns(work)
    work = validate_numeric_ranges(work, mode=mode)

    for col in columns:
        if col not in work.columns:
            work[col] = np.nan

    work = work[columns].copy()
    return work.reset_index(drop=True)

## 10) End-to-end scraper

In [11]:
def scrape_mode(mode: str, cfg: ScraperConfig) -> pd.DataFrame:
    assert mode in {"sale", "rent"}

    target = cfg.max_sale_records if mode == "sale" else cfg.max_rental_records
    session = build_session(cfg.user_agent)

    detail_urls = collect_detail_urls_for_mode(mode=mode, cfg=cfg)
    logger.info(f"Collected {len(detail_urls)} detail URLs for mode={mode}")

    rows = []

    for idx, url in enumerate(tqdm(detail_urls[:target], desc=f"Scraping {mode} details")):
        seed_property_type = None
        if mode == "sale" and "/apartments-for-sale/" in url:
            seed_property_type = "Apartment"
        elif mode == "sale" and "/houses-for-sale/" in url:
            seed_property_type = "House"
        elif mode == "rent" and "/apartments-to-rent/" in url:
            seed_property_type = "Apartment"
        elif mode == "rent" and "/houses-to-rent/" in url:
            seed_property_type = "House"

        try:
            html = fetch_html(session, url, cfg)
            row = parse_detail_page(
                html=html,
                url=url,
                mode=mode,
                seed_property_type=seed_property_type,
            )
            rows.append(row)
        except requests.HTTPError as exc:
            logger.warning(f"HTTP error on {url}: {exc}")
        except requests.RequestException as exc:
            logger.warning(f"Request error on {url}: {exc}")
        except Exception as exc:
            logger.warning(f"Parse error on {url}: {exc}")

        if (idx + 1) % cfg.detail_pause_every == 0:
            logger.info(f"Taking a longer pause after {idx + 1} detail pages")
            time.sleep(cfg.long_pause_seconds)

    raw_df = pd.DataFrame(rows)
    final_df = finalize_frame(raw_df, mode=mode)
    return final_df

## 11) Run the sales scraper

In [12]:
# Run when ready
df_listings = scrape_mode("sale", CONFIG)
df_listings.head()

17:26:59  [INFO]  ======================================================================
17:26:59  [INFO]  SALE seed: Johannesburg Metro Houses For Sale
17:26:59  [INFO]  GET https://www.privateproperty.co.za/houses-for-sale/johannesburg-metro/33
17:27:04  [INFO]  Page 1: collected 20 new links (running total=20)
17:27:04  [INFO]  GET https://www.privateproperty.co.za/houses-for-sale/johannesburg-metro/33?page=2
17:27:07  [INFO]  Page 2: collected 20 new links (running total=40)
17:27:07  [INFO]  GET https://www.privateproperty.co.za/houses-for-sale/johannesburg-metro/33?page=3
17:27:10  [INFO]  Page 3: collected 20 new links (running total=60)
17:27:10  [INFO]  GET https://www.privateproperty.co.za/houses-for-sale/johannesburg-metro/33?page=4
17:27:13  [INFO]  Page 4: collected 20 new links (running total=80)
17:27:13  [INFO]  GET https://www.privateproperty.co.za/houses-for-sale/johannesburg-metro/33?page=5
17:27:17  [INFO]  Page 5: collected 20 new links (running total=100)
17:27:17

Scraping sale details:   0%|          | 0/1000 [00:00<?, ?it/s]

17:29:51  [INFO]  GET https://www.privateproperty.co.za/for-sale/gauteng/johannesburg-metro/midrand/halfway-gardens/T5322389
17:29:56  [INFO]  GET https://www.privateproperty.co.za/for-sale/gauteng/johannesburg-metro/northcliff/linden/T5413400
17:29:59  [INFO]  GET https://www.privateproperty.co.za/for-sale/gauteng/johannesburg-metro/sandton/river-club/T5399982
17:30:02  [INFO]  GET https://www.privateproperty.co.za/for-sale/gauteng/johannesburg-metro/rosebank-and-parktown/parktown/T5419042
17:30:05  [INFO]  GET https://www.privateproperty.co.za/for-sale/gauteng/johannesburg-metro/johannesburg-central/lyndhurst/T5403283
17:30:09  [INFO]  GET https://www.privateproperty.co.za/for-sale/gauteng/johannesburg-metro/johannesburg-south/the-hill/T5244858
17:30:11  [INFO]  GET https://www.privateproperty.co.za/for-sale/gauteng/johannesburg-metro/northcliff/greenside/T5395895
17:30:15  [INFO]  GET https://www.privateproperty.co.za/for-sale/gauteng/johannesburg-metro/johannesburg-south/elandspark

,source_site,listing_id,listing_url,title,purchase_price,suburb,city,province,property_type,bedrooms,bathrooms,parking_spaces,garage,floor_area_sqm,land_area_sqm,levies,rates_taxes,description,listing_date,scraped_timestamp,pp_transaction_slug,pp_province_slug,pp_metro_slug,pp_city_slug,pp_suburb_slug
0,privateproperty,T5322389,https://www.privateproperty.co.za/for-sale/gauteng/johannesburg-metro/midrand/halfway-gardens/T5322389,3 Bedroom House in Halfway Gardens,2996000.0,Halfway Gardens,Midrand,Gauteng,House,3.0,2.5,4.0,2.0,250.0,389.0,2420.0,1751.0,"3 Bedroom House in Halfway Gardens, On Show by Appointment Only. Please contact the agent to arrange for your private viewing. This is",2 Dec 2025,2026-03-27T15:29:56.546614+00:00,for-sale,gauteng,johannesburg-metro,midrand,halfway-gardens
1,privateproperty,T5413400,https://www.privateproperty.co.za/for-sale/gauteng/johannesburg-metro/northcliff/linden/T5413400,4 Bedroom House in Linden,2475000.0,Linden,Northcliff,Gauteng,House,4.0,2.0,2.0,1.0,NaN,1.0,NaN,1996.0,"4 Bedroom House in Linden, Loved and well-maintained for many years, this warm and inviting family home is ready to welcome its",5 Mar 2026,2026-03-27T15:29:59.301851+00:00,for-sale,gauteng,johannesburg-metro,northcliff,linden
2,privateproperty,T5399982,https://www.privateproperty.co.za/for-sale/gauteng/johannesburg-metro/sandton/river-club/T5399982,3 Bedroom House in River Club,3999999.0,River Club,Sandton,Gauteng,House,3.0,2.0,8.0,2.0,350.0,1.0,NaN,2961.0,"3 Bedroom House in River Club, Sparkling and Modern with beautifully clean cut lines, this meticulously finished home is opulent in",23 Feb 2026,2026-03-27T15:30:02.255733+00:00,for-sale,gauteng,johannesburg-metro,sandton,river-club
3,privateproperty,T5419042,https://www.privateproperty.co.za/for-sale/gauteng/johannesburg-metro/rosebank-and-parktown/parktown/T5419042,4 Bedroom House in Parktown,4900000.0,Parktown,Rosebank And Parktown,Gauteng,House,4.0,2.5,9.0,3.0,470.0,1.0,NaN,4083.0,"4 Bedroom House in Parktown, Discover a home where history and lifestyle meet in perfect harmony. Designed in 1911 by W. Hoskings",10 Mar 2026,2026-03-27T15:30:05.293953+00:00,for-sale,gauteng,johannesburg-metro,rosebank-and-parktown,parktown
4,privateproperty,T5403283,https://www.privateproperty.co.za/for-sale/gauteng/johannesburg-metro/johannesburg-central/lyndhurst/T5403283,4 Bedroom House in Lyndhurst,2699000.0,Lyndhurst,Johannesburg Central,Gauteng,House,4.0,2.0,4.0,2.0,250.0,2.0,NaN,NaN,"4 Bedroom House in Lyndhurst, Tucked away in a quiet panhandle, this beautifully maintained family home offers complete privacy an",25 Feb 2026,2026-03-27T15:30:09.331485+00:00,for-sale,gauteng,johannesburg-metro,johannesburg-central,lyndhurst


In [13]:
df_listings.describe()

,purchase_price,bedrooms,bathrooms,parking_spaces,garage,floor_area_sqm,land_area_sqm,levies,rates_taxes
count,9.920000e+02,996.000000,996.000000,876.000000,423.000000,785.000000,574.000000,599.000000,808.000000
mean,2.627828e+06,2.689257,2.074297,2.846461,2.091017,166.468153,228.369338,2545.089065,1563.180160
std,3.745360e+06,1.464170,1.404766,2.511141,0.964716,184.058517,251.866095,2159.011808,1591.810995
min,1.500000e+05,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,7.499712e+05,2.000000,1.000000,1.000000,1.000000,62.000000,4.000000,1398.000000,560.000000
50%,1.250000e+06,2.000000,2.000000,2.000000,2.000000,88.000000,181.500000,2000.000000,944.000000
75%,2.881000e+06,3.000000,2.000000,3.000000,2.000000,189.000000,317.000000,2900.000000,1972.750000
max,3.000000e+07,16.000000,12.000000,18.000000,6.000000,980.000000,996.000000,16000.000000,10828.000000


In [14]:
df_listings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 996 entries, 0 to 995
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   source_site          996 non-null    object 
 1   listing_id           994 non-null    object 
 2   listing_url          996 non-null    object 
 3   title                996 non-null    object 
 4   purchase_price       992 non-null    float64
 5   suburb               996 non-null    object 
 6   city                 996 non-null    object 
 7   province             996 non-null    object 
 8   property_type        993 non-null    object 
 9   bedrooms             996 non-null    float64
 10  bathrooms            996 non-null    float64
 11  parking_spaces       876 non-null    float64
 12  garage               423 non-null    float64
 13  floor_area_sqm       785 non-null    float64
 14  land_area_sqm        574 non-null    float64
 15  levies               599 non-null    flo

## 12) Run the rental scraper

In [15]:
# Run when ready
df_rentals = scrape_mode("rent", CONFIG)
df_rentals.head()

18:32:35  [INFO]  ======================================================================
18:32:35  [INFO]  RENT seed: Johannesburg Metro Houses To Rent
18:32:35  [INFO]  GET https://www.privateproperty.co.za/houses-to-rent/johannesburg-metro/33
18:32:42  [INFO]  Page 1: collected 20 new links (running total=20)
18:32:42  [INFO]  GET https://www.privateproperty.co.za/houses-to-rent/johannesburg-metro/33?page=2
18:32:46  [INFO]  Page 2: collected 20 new links (running total=40)
18:32:46  [INFO]  GET https://www.privateproperty.co.za/houses-to-rent/johannesburg-metro/33?page=3
18:32:50  [INFO]  Page 3: collected 20 new links (running total=60)
18:32:50  [INFO]  GET https://www.privateproperty.co.za/houses-to-rent/johannesburg-metro/33?page=4
18:32:53  [INFO]  Page 4: collected 20 new links (running total=80)
18:32:53  [INFO]  GET https://www.privateproperty.co.za/houses-to-rent/johannesburg-metro/33?page=5
18:32:57  [INFO]  Page 5: collected 20 new links (running total=100)
18:32:57  [INF

Scraping rent details:   0%|          | 0/1000 [00:00<?, ?it/s]

18:35:34  [INFO]  GET https://www.privateproperty.co.za/to-rent/gauteng/johannesburg-metro/sandton/morningside/RR4635422
18:35:40  [INFO]  GET https://www.privateproperty.co.za/to-rent/gauteng/johannesburg-metro/sandton/benmore-gardens/RR4638183
18:35:42  [INFO]  GET https://www.privateproperty.co.za/to-rent/gauteng/johannesburg-metro/randburg/ferndale/RR4610204
18:35:45  [INFO]  GET https://www.privateproperty.co.za/to-rent/gauteng/johannesburg-metro/midrand/grand-central/3-tangelo-place/197-church-street/RR3307475
18:35:48  [INFO]  GET https://www.privateproperty.co.za/to-rent/gauteng/johannesburg-metro/midrand/grand-central/RR4011983
18:35:52  [INFO]  GET https://www.privateproperty.co.za/to-rent/gauteng/johannesburg-metro/johannesburg-central/bezuidenhout-valley/RR4339338
18:35:55  [INFO]  GET https://www.privateproperty.co.za/to-rent/gauteng/johannesburg-metro/sandton/fourways/fernbrook-estate/RR4637040
18:35:59  [INFO]  GET https://www.privateproperty.co.za/to-rent/gauteng/johann

,source_site,listing_id,rental_url,title,monthly_rent,suburb,city,province,property_type,bedrooms,bathrooms,parking_spaces,garage,floor_area_sqm,land_area_sqm,levies,rates_taxes,furnished_flag,availability_text,description,listing_date,scraped_timestamp,pp_transaction_slug,pp_province_slug,pp_metro_slug,pp_city_slug,pp_suburb_slug
0,privateproperty,RR4635422,https://www.privateproperty.co.za/to-rent/gauteng/johannesburg-metro/sandton/morningside/RR4635422,6 Bedroom House in Morningside,110000.0,Morningside,Sandton,Gauteng,House,6.0,7.0,1.0,1.0,887.0,2.0,NaN,NaN,NaN,Available Now,"6 Bedroom House in Morningside, This beautifully updated residence offers the perfect blend of elegance and everyday comfort. Set on",20 Mar 2026,2026-03-27T16:35:40.246438+00:00,to-rent,gauteng,johannesburg-metro,sandton,morningside
1,privateproperty,RR4638183,https://www.privateproperty.co.za/to-rent/gauteng/johannesburg-metro/sandton/benmore-gardens/RR4638183,5 Bedroom House in Benmore Gardens,75000.0,Benmore Gardens,Sandton,Gauteng,House,5.0,5.0,3.0,3.0,NaN,803.0,NaN,NaN,NaN,Available Now,"5 Bedroom House in Benmore Gardens, Waterstone Estate is located on the doorstep of Sandton, one of Johannesburg and South Africa''s mos",24 Mar 2026,2026-03-27T16:35:42.881578+00:00,to-rent,gauteng,johannesburg-metro,sandton,benmore-gardens
2,privateproperty,RR4610204,https://www.privateproperty.co.za/to-rent/gauteng/johannesburg-metro/randburg/ferndale/RR4610204,3 Bedroom House in Ferndale,19900.0,Ferndale,Randburg,Gauteng,House,3.0,2.0,5.0,2.0,205.0,NaN,NaN,NaN,NaN,Available Now,"3 Bedroom House in Ferndale, Available 1 March 2026. R19 900pm excluding all utiliities: sewerage, Pikitup by rates and taxes ac",16 Feb 2026,2026-03-27T16:35:45.804990+00:00,to-rent,gauteng,johannesburg-metro,randburg,ferndale
3,privateproperty,RR3307475,https://www.privateproperty.co.za/to-rent/gauteng/johannesburg-metro/midrand/grand-central/3-tangelo-place/197-church-street/RR3307475,1 Bedroom House in Grand Central,6500.0,Grand Central,Midrand,Gauteng,House,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Available Now,"1 Bedroom House in Grand Central, 3 Tangelo Place, 197 Church Street, Be the first to move into this stylish and contemporary 1-bedroom, 1-bathroom apartment in a brand-n",11 Jul 2025,2026-03-27T16:35:48.904997+00:00,to-rent,gauteng,johannesburg-metro,midrand,grand-central
4,privateproperty,RR4011983,https://www.privateproperty.co.za/to-rent/gauteng/johannesburg-metro/midrand/grand-central/RR4011983,1 Bedroom House in Grand Central,6300.0,Grand Central,Midrand,Gauteng,House,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Available Now,"1 Bedroom House in Grand Central, This beautiful and well-appointed 1-bedroom, 1-bathroom apartment offers secure, convenient living i",12 Dec 2025,2026-03-27T16:35:52.589116+00:00,to-rent,gauteng,johannesburg-metro,midrand,grand-central


In [16]:
df_rentals.describe()

,monthly_rent,bedrooms,bathrooms,parking_spaces,garage,floor_area_sqm,land_area_sqm,levies,rates_taxes,furnished_flag
count,999.000000,999.000000,999.000000,781.000000,372.000000,509.000000,312.000000,0.0,0.0,240.000000
mean,19644.050050,2.511562,2.028028,2.646607,2.064516,187.063065,283.291667,NaN,NaN,0.712500
std,20696.303141,1.542767,1.373677,2.039502,0.952301,194.997502,316.289828,NaN,NaN,0.453543
min,2000.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,NaN,0.000000
25%,7000.000000,1.000000,1.000000,1.000000,2.000000,50.000000,1.000000,NaN,NaN,0.000000
50%,12500.000000,2.000000,2.000000,2.000000,2.000000,95.000000,150.500000,NaN,NaN,1.000000
75%,25000.000000,3.000000,2.500000,4.000000,2.000000,274.000000,496.000000,NaN,NaN,1.000000
max,205000.000000,20.000000,20.000000,16.000000,8.000000,991.000000,996.000000,NaN,NaN,1.000000


In [17]:
df_rentals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 999 entries, 0 to 998
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   source_site          999 non-null    object 
 1   listing_id           999 non-null    object 
 2   rental_url           999 non-null    object 
 3   title                999 non-null    object 
 4   monthly_rent         999 non-null    float64
 5   suburb               999 non-null    object 
 6   city                 999 non-null    object 
 7   province             999 non-null    object 
 8   property_type        995 non-null    object 
 9   bedrooms             999 non-null    float64
 10  bathrooms            999 non-null    float64
 11  parking_spaces       781 non-null    float64
 12  garage               372 non-null    float64
 13  floor_area_sqm       509 non-null    float64
 14  land_area_sqm        312 non-null    float64
 15  levies               0 non-null      flo

## 13) Save outputs

In [18]:

# Ensure folders exist
Path("data/raw/listings").mkdir(parents=True, exist_ok=True)
Path("data/raw/rentals").mkdir(parents=True, exist_ok=True)

# Save files
df_listings.to_csv("data/raw/listings/privateproperty_sales_final.csv", index=False)
df_rentals.to_csv("data/raw/rentals/privateproperty_rentals_final.csv", index=False)

print("✅ Files saved successfully")

✅ Files saved successfully


## 14) Reload saved datasets

In [19]:
sales_path = Path("data/raw/listings/privateproperty_sales_final.csv")
rentals_path = Path("data/raw/rentals/privateproperty_rentals_final.csv")

df_listings = pd.read_csv(sales_path) if sales_path.exists() else pd.DataFrame(columns=LISTINGS_COLS)
df_rentals = pd.read_csv(rentals_path) if rentals_path.exists() else pd.DataFrame(columns=RENTALS_COLS)

print("Listings:", df_listings.shape)
print("Rentals :", df_rentals.shape)

Listings: (996, 25)
Rentals : (999, 27)


## 15) Quality report

In [20]:
def quality_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    print("=" * 70)
    print(name)
    print("=" * 70)

    if df.empty:
        print("⚠️ Dataset is empty")
        return pd.DataFrame()

    print("Rows   :", len(df))
    print("Columns:", df.shape[1])

    null_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
    report = pd.DataFrame(
        {
            "null_count": df.isna().sum(),
            "null_pct": null_pct,
        }
    ).sort_values(["null_pct", "null_count"], ascending=False)

    print("\nTop missing fields:")
    display(report.head(15))

    if "property_type" in df.columns:
        print("\nProperty type distribution:")
        display(df["property_type"].value_counts(dropna=False).to_frame("count"))

    return report

sales_quality = quality_report(df_listings, "Sales quality report")
rentals_quality = quality_report(df_rentals, "Rentals quality report")

Sales quality report
Rows   : 996
Columns: 25

Top missing fields:


,null_count,null_pct
garage,573,57.5
land_area_sqm,422,42.4
levies,397,39.9
floor_area_sqm,211,21.2
rates_taxes,188,18.9
parking_spaces,120,12.0
purchase_price,4,0.4
property_type,3,0.3
listing_id,2,0.2
bathrooms,0,0.0



Property type distribution:


,count
property_type,
House,605
Apartment,388
NaN,3


Rentals quality report
Rows   : 999
Columns: 27

Top missing fields:


,null_count,null_pct
levies,999,100.0
rates_taxes,999,100.0
furnished_flag,759,76.0
land_area_sqm,687,68.8
garage,627,62.8
floor_area_sqm,490,49.0
availability_text,435,43.5
parking_spaces,218,21.8
property_type,4,0.4
bathrooms,0,0.0



Property type distribution:


,count
property_type,
House,602
Apartment,393
NaN,4


## 16) Required field coverage

In [21]:
REQUIRED_SALE_FIELDS = [
    "listing_id",
    "purchase_price",
    "suburb",
    "city",
    "property_type",
    "bedrooms",
    "bathrooms",
    "parking_spaces",
    "garage",
]

REQUIRED_RENT_FIELDS = [
    "listing_id",
    "monthly_rent",
    "suburb",
    "city",
    "property_type",
    "bedrooms",
    "bathrooms",
    "parking_spaces",
    "garage",
]

def required_field_summary(df: pd.DataFrame, fields: list[str], name: str) -> pd.DataFrame:
    if df.empty:
        print(f"{name}: empty dataset")
        return pd.DataFrame()

    summary = pd.DataFrame(
        {
            "non_null_count": df[fields].notna().sum(),
            "null_count": df[fields].isna().sum(),
            "coverage_pct": (df[fields].notna().mean() * 100).round(1),
        }
    ).sort_values("coverage_pct", ascending=False)

    print(f"\n{name}")
    display(summary)
    return summary

sales_required = required_field_summary(df_listings, REQUIRED_SALE_FIELDS, "Sales required fields")
rentals_required = required_field_summary(df_rentals, REQUIRED_RENT_FIELDS, "Rentals required fields")


Sales required fields


,non_null_count,null_count,coverage_pct
suburb,996,0,100.0
city,996,0,100.0
bedrooms,996,0,100.0
bathrooms,996,0,100.0
listing_id,994,2,99.8
property_type,993,3,99.7
purchase_price,992,4,99.6
parking_spaces,876,120,88.0
garage,423,573,42.5



Rentals required fields


,non_null_count,null_count,coverage_pct
listing_id,999,0,100.0
monthly_rent,999,0,100.0
suburb,999,0,100.0
city,999,0,100.0
bedrooms,999,0,100.0
bathrooms,999,0,100.0
property_type,995,4,99.6
parking_spaces,781,218,78.2
garage,372,627,37.2


## 17) Rental benchmarks by suburb

In [22]:
def build_rental_benchmarks(df_rentals: pd.DataFrame) -> pd.DataFrame:
    if df_rentals.empty or "monthly_rent" not in df_rentals.columns:
        return pd.DataFrame(columns=["suburb", "avg_rent", "median_rent", "listings"])

    work = df_rentals.copy()
    work["monthly_rent"] = pd.to_numeric(work["monthly_rent"], errors="coerce")
    work["suburb"] = work["suburb"].astype("string").str.strip()

    out = (
        work.dropna(subset=["suburb", "monthly_rent"])
        .groupby("suburb", dropna=False)
        .agg(
            avg_rent=("monthly_rent", "mean"),
            median_rent=("monthly_rent", "median"),
            listings=("monthly_rent", "size"),
        )
        .reset_index()
        .sort_values(["listings", "avg_rent"], ascending=[False, False])
    )

    out["avg_rent"] = out["avg_rent"].round(2)
    out["median_rent"] = out["median_rent"].round(2)

    export_path = Path("data/raw/area_data/privateproperty_rental_benchmarks_by_suburb.csv")
    out.to_csv(export_path, index=False)
    print("✅ Saved:", export_path)
    return out

rental_benchmarks = build_rental_benchmarks(df_rentals)
rental_benchmarks.head(20)

✅ Saved: data\raw\area_data\privateproperty_rental_benchmarks_by_suburb.csv


,suburb,avg_rent,median_rent,listings
45,Dainfern,42142.86,42000.0,35
119,Morningside,35453.12,25500.0,32
24,Bryanston,27872.90,21000.0,31
5,Bedfordview,21975.81,24000.0,31
125,Noordwyk,11118.23,9500.0,31
59,Fourways,24285.71,25000.0,28
82,Johannesburg Cbd,4085.57,3700.0,28
185,Waterfall Estate,29514.81,22000.0,27
55,Ferndale,10869.40,7450.0,25
77,Hyde Park,62818.18,52500.0,22


## 18) Market signals by suburb

In [23]:
def build_market_signals(df_listings: pd.DataFrame, df_rentals: pd.DataFrame) -> pd.DataFrame:
    if df_listings.empty and df_rentals.empty:
        return pd.DataFrame()

    sales = df_listings.copy()
    rents = df_rentals.copy()

    if not sales.empty:
        sales["purchase_price"] = pd.to_numeric(sales["purchase_price"], errors="coerce")
        sales_summary = (
            sales.dropna(subset=["suburb", "purchase_price"])
            .groupby("suburb")
            .agg(
                avg_purchase_price=("purchase_price", "mean"),
                median_purchase_price=("purchase_price", "median"),
                sale_listings=("purchase_price", "size"),
            )
            .reset_index()
        )
    else:
        sales_summary = pd.DataFrame(columns=["suburb", "avg_purchase_price", "median_purchase_price", "sale_listings"])

    if not rents.empty:
        rents["monthly_rent"] = pd.to_numeric(rents["monthly_rent"], errors="coerce")
        rent_summary = (
            rents.dropna(subset=["suburb", "monthly_rent"])
            .groupby("suburb")
            .agg(
                avg_monthly_rent=("monthly_rent", "mean"),
                median_monthly_rent=("monthly_rent", "median"),
                rental_listings=("monthly_rent", "size"),
            )
            .reset_index()
        )
    else:
        rent_summary = pd.DataFrame(columns=["suburb", "avg_monthly_rent", "median_monthly_rent", "rental_listings"])

    out = sales_summary.merge(rent_summary, on="suburb", how="outer")

    if {"avg_purchase_price", "avg_monthly_rent"}.issubset(out.columns):
        out["gross_yield_pct_est"] = ((out["avg_monthly_rent"] * 12) / out["avg_purchase_price"] * 100).round(2)

    export_path = Path("data/raw/area_data/privateproperty_market_signals_by_suburb.csv")
    out.to_csv(export_path, index=False)
    print("✅ Saved:", export_path)

    sort_cols = [c for c in ["sale_listings", "rental_listings"] if c in out.columns]
    if sort_cols:
        out = out.sort_values(sort_cols, ascending=False)

    return out

market_signals = build_market_signals(df_listings, df_rentals)
market_signals.head(20)

✅ Saved: data\raw\area_data\privateproperty_market_signals_by_suburb.csv


,suburb,avg_purchase_price,median_purchase_price,sale_listings,avg_monthly_rent,median_monthly_rent,rental_listings,gross_yield_pct_est
183,Protea Glen,8.496220e+05,765000.0,46.0,4961.538462,5000.0,13.0,7.01
31,Bryanston,5.354204e+06,3299000.0,44.0,27872.903226,21000.0,31.0,6.25
73,Ferndale,8.927187e+05,689999.0,35.0,10869.400000,7450.0,25.0,14.61
146,Morningside,3.338715e+06,2099500.0,30.0,35453.125000,25500.0,32.0,12.74
129,Lonehill,2.117923e+06,1924500.0,26.0,23966.666667,21500.0,12.0,13.58
77,Fourways,2.245542e+06,1547500.0,24.0,24285.714286,25000.0,28.0,12.98
178,Paulshof,1.337522e+06,949999.0,23.0,14399.857143,8000.0,7.0,12.92
156,Noordwyk,1.008102e+06,900000.0,21.0,11118.225806,9500.0,31.0,13.23
228,Waterfall Estate,5.934947e+06,2800000.0,19.0,29514.814815,22000.0,27.0,5.97
158,North Riding,1.169474e+06,999000.0,19.0,14642.769231,8990.0,13.0,15.02


## 19) Final export check

In [24]:
exports = [
    "data/raw/listings/privateproperty_sales_final.csv",
    "data/raw/rentals/privateproperty_rentals_final.csv",
    "data/raw/area_data/privateproperty_rental_benchmarks_by_suburb.csv",
    "data/raw/area_data/privateproperty_market_signals_by_suburb.csv",
]

print("Expected exports:")
for path in exports:
    print(" -", path, "✅" if Path(path).exists() else "⏳")

Expected exports:
 - data/raw/listings/privateproperty_sales_final.csv ✅
 - data/raw/rentals/privateproperty_rentals_final.csv ✅
 - data/raw/area_data/privateproperty_rental_benchmarks_by_suburb.csv ✅
 - data/raw/area_data/privateproperty_market_signals_by_suburb.csv ✅
